# N106 · 状态机DP与股票交易

**目标：** 用持有状态和交易次数表达合法历史。

**先修：** N098, N099, N023。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 买卖状态；费用；冷冻期；K次交易；不可达初始化；同日更新顺序。

## 从问题到算法

股票DP是带约束的状态机。状态必须说清持仓、已用交易次数与是否处于冷冻期；当天转移应读前一天的状态，避免无意中在同一天多次操作。费用只扣一次。所有接口允许不交易，最终只能返回未持仓收益。

## 最小实现

**本章接口：** `max_profit_once(prices)`、`max_profit_k(prices, k)`、`max_profit_cooldown(prices)`、`max_profit_fee(prices, fee)`

In [1]:
def max_profit_once(prices):
    low=float('inf'); best=0
    for p in prices: low=min(low,p); best=max(best,p-low)
    return best

def max_profit_k(prices,k):
    if k<=0: return 0
    if k>=len(prices)//2: return sum(max(0,b-a) for a,b in zip(prices,prices[1:]))
    buy=[float('-inf')]*(k+1); sell=[0]*(k+1)
    for p in prices:
        old_buy,old_sell=buy,sell; buy=old_buy[:]; sell=old_sell[:]
        for t in range(1,k+1):
            buy[t]=max(old_buy[t],old_sell[t-1]-p)
            sell[t]=max(old_sell[t],old_buy[t]+p)
    return sell[k]

def max_profit_cooldown(prices):
    hold=sold=float('-inf'); rest=0
    for p in prices: hold,sold,rest=max(hold,rest-p),hold+p,max(rest,sold)
    return max(0,sold,rest)

def max_profit_fee(prices,fee):
    hold=float('-inf'); cash=0
    for p in prices: hold,cash=max(hold,cash-p),max(cash,hold+p-fee)
    return cash

## 正确性、前提与复杂度

每个合法交易过程映射到状态机的一条路径。枚举休息、买入、卖出三种动作完整覆盖合法路径，非法转移被状态限制排除。冷冻状态只能在卖出后经过一天休息再买；元组赋值右侧都读取旧值。

**代价：** 单次、冷冻和费用版O(n)时间O(1)辅助；k次版O(nk)时间O(k)辅助，k足够大时退化为无限交易O(n)。价格非负，手续费非负，交易必须先买后卖。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

prices=[1,2,3,0,2]; hold=sold=float('-inf'); rest=0; rows=[]
for day,p in enumerate(prices):
    hold,sold,rest=max(hold,rest-p),hold+p,max(rest,sold); rows.append((day,p,hold,sold,rest))
show_table(['天','价格','持仓','刚售出','可买入'],rows)

天,价格,持仓,刚售出,可买入
0,1,-1,-inf,0
1,2,-1,1,0
2,3,-1,2,1
3,0,1,-1,2
4,2,1,3,2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert max_profit_once([7,1,5,3,6,4])==5
assert max_profit_cooldown([1,2,3,0,2])==3
assert max_profit_fee([1,3,2,8,4,9],2)==8
from itertools import product
from functools import cache

def brute(prices,k=99,fee=0,cooldown=False):
    @cache
    def f(i,holding,remaining,blocked):
        if i==len(prices): return float('-inf') if holding else 0
        best=f(i+1,holding,remaining,False)
        if holding and remaining: best=max(best,prices[i]-fee+f(i+1,False,remaining-1,cooldown))
        if not holding and not blocked and remaining: best=max(best,-prices[i]+f(i+1,True,remaining,False))
        return best
    return f(0,False,k,False)
for prices in product(range(3),repeat=5):
    assert max_profit_once(prices)==brute(prices,1)
    assert max_profit_k(prices,2)==brute(prices,2)
    assert max_profit_fee(prices,1)==brute(prices,fee=1)
    assert max_profit_cooldown(prices)==brute(prices,cooldown=True)
print('N106: 所有本章断言通过')

N106: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 从统一状态机派生无限次交易。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释初始持有不能初始化为0。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 121. Best Time to Buy and Sell Stock（canonical）
- 122. Best Time to Buy and Sell Stock II（canonical）
- 188. Best Time to Buy and Sell Stock IV（canonical）
- 309. Best Time to Buy and Sell Stock with Cooldown（canonical）
- 714. Best Time to Buy and Sell Stock with Transaction Fee（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。